# PortWatch Silver Fact: Monthly Chokepoint Stress

This notebook builds a monthly chokepoint stress mart from PortWatch daily bronze parquet.

Outputs include:
- `avg_n_total`, `avg_capacity`, `max_n_total`, `max_capacity`
- vessel mix shares (`tanker`, `container`, `dry_bulk`)
- z-scores for `n_total` and `capacity`
- `stress_index` and optional `stress_index_weighted`
- monthly parquet files in silver storage partitions (`year=YYYY/month=MM`)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upward until pyproject.toml is found, then return that folder."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


ROOT = find_project_root(Path.cwd())
BRONZE_PORTWATCH = ROOT / "data" / "bronze" / "portwatch"
SILVER_PORTWATCH = ROOT / "data" / "silver" / "portwatch"
MONTHLY_MART_DIR = SILVER_PORTWATCH / "mart_portwatch_chokepoint_stress_monthly"

print(f"ROOT: {ROOT}")
print(f"BRONZE_PORTWATCH: {BRONZE_PORTWATCH}")
print(f"MONTHLY_MART_DIR: {MONTHLY_MART_DIR}")

# Read all daily parquet files and concatenate to avoid partition dtype merge conflicts.
parquet_files = sorted(BRONZE_PORTWATCH.rglob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found under {BRONZE_PORTWATCH}")

df = pd.concat((pd.read_parquet(p) for p in parquet_files), ignore_index=True)
print(f"Loaded files: {len(parquet_files):,}")
print(f"Loaded rows: {len(df):,}")
print(f"Columns: {sorted(df.columns.tolist())}")

# Type cleanup
if not np.issubdtype(df["date"].dtype, np.datetime64):
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

numeric_cols = [
    "n_total", "capacity",
    "n_tanker", "n_container", "n_dry_bulk",
    "capacity_tanker", "capacity_container", "capacity_dry_bulk",
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Keep only rows with required fields present
df = df.dropna(subset=["date", "portname", "n_total", "capacity"]).copy()
df["year_month"] = df["date"].dt.to_period("M").astype(str)

# Optional: keep the key chokepoints you identified for this first mart.
target_chokepoints = {
    "Suez Canal",
    "Strait of Hormuz",
    "Panama Canal",
    "Cape of Good Hope",
    "Bab el-Mandeb Strait",
}
df = df[df["portname"].isin(target_chokepoints)].copy()

print("Rows after cleaning/filtering:", f"{len(df):,}")
print("Chokepoints:", sorted(df["portname"].dropna().unique().tolist()))
print("Date range:", df["date"].min(), "->", df["date"].max())

ROOT: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly
BRONZE_PORTWATCH: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/bronze/portwatch
MONTHLY_MART_DIR: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/mart_portwatch_chokepoint_stress_monthly
Loaded files: 2,259
Loaded rows: 5,782
Columns: ['ObjectId', 'capacity', 'capacity_cargo', 'capacity_container', 'capacity_dry_bulk', 'capacity_general_cargo', 'capacity_roro', 'capacity_tanker', 'date', 'day', 'month', 'n_cargo', 'n_container', 'n_dry_bulk', 'n_general_cargo', 'n_roro', 'n_tanker', 'n_total', 'portid', 'portname', 'year']
Rows after cleaning/filtering: 5,782
Chokepoints: ['Bab el-Mandeb Strait', 'Cape of Good Hope', 'Panama Canal', 'Strait of Hormuz', 'Suez Canal']
Date range: 2020-01-01 00:00:00 -> 2026-03-08 00:00:00


In [2]:
# Monthly mart at grain: chokepoint x month
monthly = (
    df.groupby(["portname", "year_month"], as_index=False)
      .agg(
          avg_n_total=("n_total", "mean"),
          max_n_total=("n_total", "max"),
          avg_capacity=("capacity", "mean"),
          max_capacity=("capacity", "max"),
          avg_n_tanker=("n_tanker", "mean"),
          avg_n_container=("n_container", "mean"),
          avg_n_dry_bulk=("n_dry_bulk", "mean"),
          avg_capacity_tanker=("capacity_tanker", "mean"),
          avg_capacity_container=("capacity_container", "mean"),
          avg_capacity_dry_bulk=("capacity_dry_bulk", "mean"),
          days_observed=("date", "nunique"),
      )
      .rename(columns={"portname": "chokepoint_name"})
)

# Vessel shares useful for weighted downstream scenarios.
monthly["total_vessel_classes_avg"] = (
    monthly["avg_n_tanker"].fillna(0)
    + monthly["avg_n_container"].fillna(0)
    + monthly["avg_n_dry_bulk"].fillna(0)
)
monthly["tanker_share"] = np.where(
    monthly["total_vessel_classes_avg"] > 0,
    monthly["avg_n_tanker"] / monthly["total_vessel_classes_avg"],
    np.nan,
)
monthly["container_share"] = np.where(
    monthly["total_vessel_classes_avg"] > 0,
    monthly["avg_n_container"] / monthly["total_vessel_classes_avg"],
    np.nan,
)
monthly["dry_bulk_share"] = np.where(
    monthly["total_vessel_classes_avg"] > 0,
    monthly["avg_n_dry_bulk"] / monthly["total_vessel_classes_avg"],
    np.nan,
)

# z-score helper per chokepoint time series
def zscore_by_group(series: pd.Series) -> pd.Series:
    mean = series.mean(skipna=True)
    std = series.std(skipna=True, ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(series), dtype=float), index=series.index)
    return (series - mean) / std

monthly = monthly.sort_values(["chokepoint_name", "year_month"]).copy()
monthly["z_n_total"] = monthly.groupby("chokepoint_name")["avg_n_total"].transform(zscore_by_group)
monthly["z_capacity"] = monthly.groupby("chokepoint_name")["avg_capacity"].transform(zscore_by_group)

# Requested first stress index: average of n_total and capacity z-scores.
monthly["stress_index"] = 0.5 * monthly["z_n_total"] + 0.5 * monthly["z_capacity"]

# Optional weighted stress by chokepoint-vessel sensitivity.
monthly["priority_vessel_share"] = np.select(
    [
        monthly["chokepoint_name"].eq("Strait of Hormuz"),
        monthly["chokepoint_name"].isin(["Suez Canal", "Panama Canal"]),
    ],
    [
        monthly["tanker_share"],
        monthly["container_share"],
    ],
    default=monthly["dry_bulk_share"],
)
monthly["stress_index_weighted"] = monthly["stress_index"] * (
    1.0 + 0.5 * monthly["priority_vessel_share"].fillna(0.0)
)

# Partition columns for month-wise parquet writes
period_index = pd.PeriodIndex(monthly["year_month"], freq="M")
monthly["year"] = period_index.year.astype(int)
monthly["month"] = period_index.month.astype(int).astype(str).str.zfill(2)

monthly_cols = [
    "year_month",
    "chokepoint_name",
    "avg_n_total",
    "max_n_total",
    "avg_capacity",
    "max_capacity",
    "avg_n_tanker",
    "avg_n_container",
    "avg_n_dry_bulk",
    "avg_capacity_tanker",
    "avg_capacity_container",
    "avg_capacity_dry_bulk",
    "days_observed",
    "stress_index",
    "stress_index_weighted",
    "tanker_share",
    "container_share",
    "dry_bulk_share",
    "year",
    "month",
]
monthly = monthly[monthly_cols]

monthly.head(12)

,year_month,chokepoint_name,avg_n_total,max_n_total,avg_capacity,max_capacity,avg_n_tanker,avg_n_container,avg_n_dry_bulk,avg_capacity_tanker,avg_capacity_container,avg_capacity_dry_bulk,days_observed,stress_index,stress_index_weighted,tanker_share,container_share,dry_bulk_share,year,month
0,2020-06,Bab el-Mandeb Strait,49.500000,52,2.275656e+06,2279674,13.500000,14.000000,11.000000,647693.000000,1.082319e+06,467936.500000,2,-1.135328,-1.297518,0.350649,0.363636,0.285714,2020,06
1,2020-07,Bab el-Mandeb Strait,52.354839,65,2.338218e+06,2941222,17.290323,14.516129,12.354839,689030.903226,9.817766e+05,629520.322581,31,-0.940423,-1.071972,0.391527,0.328707,0.279766,2020,07
2,2020-08,Bab el-Mandeb Strait,56.666667,62,2.533425e+06,2899512,15.666667,18.111111,15.888889,645923.555556,1.102497e+06,746976.555556,9,-0.554484,-0.643176,0.315436,0.364653,0.319911,2020,08
3,2020-09,Bab el-Mandeb Strait,55.368421,68,2.480720e+06,3066856,17.105263,16.105263,15.578947,650206.210526,1.035707e+06,765896.684211,19,-0.665168,-0.771365,0.350593,0.330097,0.319310,2020,09
4,2020-10,Bab el-Mandeb Strait,52.947368,81,2.377769e+06,3332430,15.947368,15.578947,15.000000,635236.263158,1.039268e+06,671508.684211,19,-0.875818,-1.016999,0.342760,0.334842,0.322398,2020,10
5,2020-11,Bab el-Mandeb Strait,55.272727,64,2.332337e+06,2929734,16.454545,16.363636,15.272727,586498.454545,1.034215e+06,682847.181818,11,-0.804692,-0.932469,0.342155,0.340265,0.317580,2020,11
6,2020-12,Bab el-Mandeb Strait,54.785714,65,2.472469e+06,3278193,16.928571,15.857143,14.321429,666134.821429,1.045706e+06,721432.035714,28,-0.700842,-0.807376,0.359363,0.336619,0.304018,2020,12
7,2021-03,Bab el-Mandeb Strait,50.681818,66,2.267096e+06,3459568,16.727273,14.272727,13.000000,700802.727273,8.629249e+05,668812.636364,22,-1.085970,-1.246398,0.380165,0.324380,0.295455,2021,03
8,2021-04,Bab el-Mandeb Strait,61.428571,71,2.943974e+06,3975415,18.857143,18.428571,15.571429,792614.857143,1.197985e+06,900513.571429,7,0.048987,0.056203,0.356757,0.348649,0.294595,2021,04
9,2021-06,Bab el-Mandeb Strait,60.833333,71,2.885253e+06,3532448,18.000000,17.333333,15.833333,810990.333333,1.098668e+06,936228.500000,12,-0.033177,-0.038310,0.351792,0.338762,0.309446,2021,06


In [3]:
# Persist to silver: one parquet file per month partition for cloud/blob-friendly layout
MONTHLY_MART_DIR.mkdir(parents=True, exist_ok=True)

written_files = []
for (y, m), part in monthly.groupby(["year", "month"], sort=True):
    out_dir = MONTHLY_MART_DIR / f"year={int(y)}" / f"month={m}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "portwatch_chokepoint_stress_monthly.parquet"
    part.sort_values(["chokepoint_name", "year_month"]).to_parquet(out_path, index=False)
    written_files.append(str(out_path))

# Optional full-table convenience file for quick reads
full_out = SILVER_PORTWATCH / "portwatch_chokepoint_stress_monthly_all.parquet"
monthly.sort_values(["year_month", "chokepoint_name"]).to_parquet(full_out, index=False)

# Suggested downstream dimensions from this same PortWatch layer
dim_chokepoint = (
    df.groupby(["portid", "portname"], as_index=False)
      .agg(
          first_observed_date=("date", "min"),
          last_observed_date=("date", "max"),
          days_observed_total=("date", "nunique"),
          mean_n_total=("n_total", "mean"),
          mean_capacity=("capacity", "mean"),
      )
      .rename(columns={"portid": "chokepoint_id", "portname": "chokepoint_name"})
)

share_rollup = (
    monthly.groupby("chokepoint_name", as_index=False)
    .agg(
        avg_tanker_share=("tanker_share", "mean"),
        avg_container_share=("container_share", "mean"),
        avg_dry_bulk_share=("dry_bulk_share", "mean"),
    )
)

dim_chokepoint = dim_chokepoint.merge(share_rollup, on="chokepoint_name", how="left")
dim_chokepoint["dominant_vessel_class"] = (
    dim_chokepoint[["avg_tanker_share", "avg_container_share", "avg_dry_bulk_share"]]
    .idxmax(axis=1)
    .str.replace("avg_", "", regex=False)
    .str.replace("_share", "", regex=False)
)

dim_month = (
    monthly[["year_month", "year", "month"]]
    .drop_duplicates()
    .assign(month_start=lambda x: pd.PeriodIndex(x["year_month"], freq="M").to_timestamp())
    .sort_values("year_month")
)

dim_dir = SILVER_PORTWATCH / "dimensions"
dim_dir.mkdir(parents=True, exist_ok=True)
dim_chokepoint_out = dim_dir / "dim_portwatch_chokepoint.parquet"
dim_month_out = dim_dir / "dim_month.parquet"
dim_chokepoint.to_parquet(dim_chokepoint_out, index=False)
dim_month.to_parquet(dim_month_out, index=False)

print(f"Wrote {len(written_files)} monthly partition files")
print(f"Combined table: {full_out}")
print(f"Dimension table (chokepoint): {dim_chokepoint_out}")
print(f"Dimension table (month): {dim_month_out}")
print("Sample partitions:")
for p in written_files[:5]:
    print(" -", p)

monthly.tail(12)

Wrote 75 monthly partition files
Combined table: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/portwatch_chokepoint_stress_monthly_all.parquet
Dimension table (chokepoint): /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/dimensions/dim_portwatch_chokepoint.parquet
Dimension table (month): /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/dimensions/dim_month.parquet
Sample partitions:
 - /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/mart_portwatch_chokepoint_stress_monthly/year=2020/month=01/portwatch_chokepoint_stress_monthly.parquet
 - /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/mart_portwatch_chokepoint_stress_monthly/year=2020/month=02/portwatch_chokepoint_stress_monthly.parquet
 - /Users/chromazone/Documents/Python/Data Engi

,year_month,chokepoint_name,avg_n_total,max_n_total,avg_capacity,max_capacity,avg_n_tanker,avg_n_container,avg_n_dry_bulk,avg_capacity_tanker,avg_capacity_container,avg_capacity_dry_bulk,days_observed,stress_index,stress_index_weighted,tanker_share,container_share,dry_bulk_share,year,month
215,2025-04,Suez Canal,36.733333,59,1.308745e+06,2128256,14.300000,9.033333,8.833333,658959.233333,180296.533333,447021.666667,30,-1.052638,-1.200444,0.444560,0.280829,0.274611,2025,04
216,2025-05,Suez Canal,36.935484,51,1.338087e+06,2252164,14.387097,9.290323,9.000000,681537.129032,193430.161290,436122.709677,31,-1.018323,-1.163080,0.440276,0.284304,0.275420,2025,05
217,2025-06,Suez Canal,36.466667,47,1.228664e+06,1849434,14.466667,9.200000,8.166667,638863.200000,195867.066667,371158.200000,30,-1.130887,-1.294303,0.454450,0.289005,0.256545,2025,06
218,2025-07,Suez Canal,37.645161,50,1.385633e+06,2452111,15.290323,8.483871,8.935484,727225.709677,180008.838710,456334.032258,31,-0.942070,-1.064241,0.467456,0.259369,0.273176,2025,07
219,2025-08,Suez Canal,38.387097,60,1.364763e+06,2100152,14.806452,9.032258,9.967742,652541.193548,187448.419355,501389.935484,31,-0.918612,-1.041328,0.437977,0.267176,0.294847,2025,08
220,2025-09,Suez Canal,41.700000,52,1.510670e+06,2047555,15.866667,9.533333,11.433333,744927.500000,204382.833333,536050.700000,30,-0.623269,-0.703927,0.430769,0.258824,0.310407,2025,09
221,2025-10,Suez Canal,41.032258,51,1.512137e+06,2193370,15.322581,10.000000,10.903226,750141.064516,199387.354839,538416.967742,31,-0.658185,-0.749030,0.422974,0.276046,0.300980,2025,10
222,2025-11,Suez Canal,42.500000,52,1.577757e+06,2717072,15.033333,9.500000,12.266667,758210.666667,186438.500000,601676.200000,30,-0.526557,-0.594523,0.408514,0.258152,0.333333,2025,11
223,2025-12,Suez Canal,39.096774,52,1.540382e+06,2276149,14.741935,8.838710,10.935484,738409.000000,216434.096774,564098.064516,31,-0.740263,-0.835045,0.427103,0.256075,0.316822,2025,12
224,2026-01,Suez Canal,37.677419,50,1.300326e+06,1779612,13.580645,8.967742,9.516129,632746.580645,201838.548387,438473.064516,31,-1.008330,-1.149334,0.423541,0.279678,0.296781,2026,01


In [4]:
# Quick QA checks
qa = (
    monthly.groupby("chokepoint_name", as_index=False)
    .agg(
        months=("year_month", "nunique"),
        avg_stress=("stress_index", "mean"),
        max_stress=("stress_index", "max"),
        min_stress=("stress_index", "min"),
    )
    .sort_values("max_stress", ascending=False)
)

qa

,chokepoint_name,months,avg_stress,max_stress,min_stress
4,Suez Canal,46,-2.413528e-17,1.873889,-1.302677
0,Bab el-Mandeb Strait,39,-1.138690e-17,1.428241,-3.092510
1,Cape of Good Hope,50,-1.887379e-16,1.380582,-1.619699
2,Panama Canal,43,-4.750722e-16,1.293221,-2.712759
3,Strait of Hormuz,49,6.162871e-16,1.128184,-4.858020


In [5]:
# Month x chokepoint scaffold + coverage flags
# This creates a complete grid, then left joins the monthly fact.
scaffold = (
    dim_month[["year_month", "year", "month"]]
    .drop_duplicates()
    .merge(
        dim_chokepoint[["chokepoint_id", "chokepoint_name"]].drop_duplicates(),
        how="cross",
    )
)

fact_cols = [
    "year_month",
    "chokepoint_name",
    "avg_n_total",
    "max_n_total",
    "avg_capacity",
    "max_capacity",
    "avg_n_tanker",
    "avg_n_container",
    "avg_n_dry_bulk",
    "avg_capacity_tanker",
    "avg_capacity_container",
    "avg_capacity_dry_bulk",
    "days_observed",
    "stress_index",
    "stress_index_weighted",
]

portwatch_scaffold = scaffold.merge(
    monthly[fact_cols],
    on=["year_month", "chokepoint_name"],
    how="left",
)

# Flags requested for downstream QA and completeness checks
portwatch_scaffold["has_portwatch_data_flag"] = (
    portwatch_scaffold["days_observed"].fillna(0).gt(0).astype("int8")
)
portwatch_scaffold["coverage_gap_flag"] = (
    portwatch_scaffold["has_portwatch_data_flag"].eq(0).astype("int8")
)

# Optional coverage ratio by row (days observed / days in month)
month_start = pd.PeriodIndex(portwatch_scaffold["year_month"], freq="M").to_timestamp()
portwatch_scaffold["days_in_month"] = month_start.days_in_month.astype("int16")
portwatch_scaffold["coverage_ratio"] = (
    portwatch_scaffold["days_observed"].fillna(0) / portwatch_scaffold["days_in_month"]
)

portwatch_scaffold = portwatch_scaffold.sort_values(["year_month", "chokepoint_name"]).reset_index(drop=True)

# Persist scaffold table to silver
scaffold_out = SILVER_PORTWATCH / "portwatch_month_chokepoint_scaffold.parquet"
portwatch_scaffold.to_parquet(scaffold_out, index=False)

print(f"Scaffold rows: {len(portwatch_scaffold):,}")
print(f"Rows with data: {int(portwatch_scaffold['has_portwatch_data_flag'].sum()):,}")
print(f"Coverage gaps: {int(portwatch_scaffold['coverage_gap_flag'].sum()):,}")
print(f"Saved scaffold: {scaffold_out}")

portwatch_scaffold.head(12)

Scaffold rows: 375
Rows with data: 227
Coverage gaps: 148
Saved scaffold: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/portwatch/portwatch_month_chokepoint_scaffold.parquet


,year_month,year,month,chokepoint_id,chokepoint_name,avg_n_total,max_n_total,avg_capacity,max_capacity,avg_n_tanker,...,avg_capacity_tanker,avg_capacity_container,avg_capacity_dry_bulk,days_observed,stress_index,stress_index_weighted,has_portwatch_data_flag,coverage_gap_flag,days_in_month,coverage_ratio
0,2020-01,2020,01,chokepoint4,Bab el-Mandeb Strait,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1,31,0.000000
1,2020-01,2020,01,chokepoint7,Cape of Good Hope,37.727273,51.0,2.127087e+06,3123806.0,9.727273,...,6.984125e+05,1.473175e+05,1.250678e+06,11.0,-1.619699,-2.077075,1,0,31,0.354839
2,2020-01,2020,01,chokepoint2,Panama Canal,31.225806,36.0,8.271396e+05,1035576.0,12.870968,...,2.884856e+05,2.311448e+05,2.816883e+05,31.0,0.294271,0.333601,1,0,31,1.000000
3,2020-01,2020,01,chokepoint6,Strait of Hormuz,66.935484,83.0,2.076516e+06,3316620.0,39.806452,...,1.317263e+06,3.788396e+05,3.595543e+05,31.0,-1.536360,-2.029306,1,0,31,1.000000
4,2020-01,2020,01,chokepoint1,Suez Canal,53.354839,70.0,2.498368e+06,3479062.0,16.645161,...,7.762834e+05,1.010523e+06,6.786981e+05,31.0,0.793924,0.939661,1,0,31,1.000000
5,2020-02,2020,02,chokepoint4,Bab el-Mandeb Strait,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1,29,0.000000
6,2020-02,2020,02,chokepoint7,Cape of Good Hope,43.315789,55.0,2.242601e+06,3910779.0,11.421053,...,9.742759e+05,1.437946e+05,1.099581e+06,19.0,-1.438842,-1.859369,1,0,29,0.655172
7,2020-02,2020,02,chokepoint2,Panama Canal,31.722222,36.0,9.102232e+05,1202192.0,13.166667,...,3.560762e+05,2.099320e+05,3.285361e+05,18.0,0.714701,0.798246,1,0,29,0.620690
8,2020-02,2020,02,chokepoint6,Strait of Hormuz,67.068966,97.0,2.606358e+06,4233521.0,40.758621,...,1.895117e+06,3.571295e+05,3.335086e+05,29.0,-1.140575,-1.512378,1,0,29,1.000000
9,2020-02,2020,02,chokepoint1,Suez Canal,53.241379,65.0,2.533162e+06,3439411.0,16.379310,...,8.528261e+05,1.007607e+06,6.430977e+05,29.0,0.815529,0.959552,1,0,29,1.000000


## Notes for Gold Linking

This monthly mart is now ready to join later with:
- route exposure bridge (`country/route -> chokepoint`)
- disruption events table (`event periods by chokepoint`)

Typical key for joins: `chokepoint_name` + `year_month`.